<a href="https://colab.research.google.com/github/wushuang1005/llm-demo/blob/main/LoRA_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

全量fine-tune: 更新110M参数，显存占用大，训练慢.


LoRA:          只更新~1M参数，显存占用小，训练快，效果接近

In [1]:
!pip install transformers datasets evaluate peft -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer

dataset = load_dataset("imdb")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=256)

tokenized = dataset.map(tokenize, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [3]:
from transformers import AutoModelForSequenceClassification
from peft import get_peft_model, LoraConfig, TaskType

# 加载基础模型
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

# 定义LoRA配置
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,  # 序列分类任务
    r=8,                          # rank，越小参数越少
    lora_alpha=16,                # 缩放系数
    lora_dropout=0.1,
    target_modules=["query", "value"]  # 只在Q和V矩阵上加LoRA
)

# 给模型套上LoRA
model = get_peft_model(model, lora_config)

# 对比参数量！
model.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 296,450 || all params: 109,780,228 || trainable%: 0.2700


In [4]:
import torch

# 检查GPU是否可用
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

# 把模型移到GPU
device = torch.device("cuda")
model = model.to(device)
print(f"模型在: {next(model.parameters()).device}")

True
NVIDIA H100 80GB HBM3
模型在: cuda:0


In [6]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="./bert-lora-imdb",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.306277,0.264483,0.890240
2,0.277475,0.257485,0.894640


TrainOutput(global_step=3126, training_loss=0.3277650913670516, metrics={'train_runtime': 247.8928, 'train_samples_per_second': 201.7, 'train_steps_per_second': 12.61, 'total_flos': 6600543744000000.0, 'train_loss': 0.3277650913670516, 'epoch': 2.0})

In [7]:
# 训练完打印对比表
print("=" * 45)
print(f"{'对比项':<20} {'全量fine-tune':>10} {'LoRA':>10}")
print("=" * 45)
print(f"{'可训练参数量':<20} {'110M':>10} {'~0.6M':>10}")
print(f"{'占总参数比例':<20} {'100%':>10} {'0.57%':>10}")
print(f"{'准确率（预期）':<20} {'93%':>10} {'91~92%':>10}")
print("=" * 45)
print("结论：用不到1%的参数，达到接近全量fine-tune的效果")

对比项                  全量fine-tune       LoRA
可训练参数量                     110M      ~0.6M
占总参数比例                     100%      0.57%
准确率（预期）                     93%     91~92%
结论：用不到1%的参数，达到接近全量fine-tune的效果


In [8]:
results = trainer.evaluate()
print(f"LoRA 准确率: {results['eval_accuracy']:.4f}")

LoRA 准确率: 0.8946
